In [ ]:
from pathlib import Path

data_dir = Path("data")

print("Current folder:", Path.cwd())

for split in ["Training", "Testing"]:
    folder = data_dir / split
    print(f"\n{split} folder found: {folder.exists()}")

    if folder.exists():
        for class_folder in sorted(folder.iterdir()):
            if class_folder.is_dir():
                image_count = sum(
                    1
                    for file in class_folder.rglob("*")
                    if file.suffix.lower() in {
                        ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"
                    }
                )
                print(f"  {class_folder.name}: {image_count} images") 

In [ ]:
import random
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from collections import Counter

# 1. Fixed seed: makes the split repeatable
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Keep the same image preprocessing as your original model
tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    )
])

# 3. Load the existing folders
full_train_ds = datasets.ImageFolder(
    "data/Training", transform=tf
)

test_ds = datasets.ImageFolder(
    "data/Testing", transform=tf
)

assert full_train_ds.class_to_idx == test_ds.class_to_idx, (
    "Training and testing class labels do not match."
)

class_names = full_train_ds.classes

# 4. Reserve 20% of EACH class for validation
rng = random.Random(SEED)
train_indices = []
val_indices = []

for class_id in range(len(class_names)):
    indices = [
        i for i, label in enumerate(full_train_ds.targets)
        if label == class_id
    ]

    rng.shuffle(indices)
    val_count = round(len(indices) * 0.20)

    val_indices.extend(indices[:val_count])
    train_indices.extend(indices[val_count:])

train_ds = Subset(full_train_ds, train_indices)
val_ds = Subset(full_train_ds, val_indices)

# Confirm that no image index belongs to both sets
assert set(train_indices).isdisjoint(val_indices)

# 5. Prepare batches of 32 images
train_dl = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED)
)

val_dl = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

test_dl = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

# 6. Display the results
print("Device:", device)
print("Classes:", class_names)
print("\nTraining images:", len(train_ds))
print("Validation images:", len(val_ds))
print("Testing images:", len(test_ds))

train_counts = Counter(
    full_train_ds.targets[i] for i in train_indices
)
val_counts = Counter(
    full_train_ds.targets[i] for i in val_indices
)

print("\nPer-class split:")
for class_id, name in enumerate(class_names):
    print(
        f"{name}: "
        f"train={train_counts[class_id]}, "
        f"validation={val_counts[class_id]}"
    )

In [ ]:
from torch import nn

def build_model(dropout=0.5):
    model = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Flatten(),
        nn.Linear(128 * 16 * 16, 256),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(256, len(class_names))
    )

    return model.to(device)


# Create a fresh model
torch.manual_seed(SEED)
model = build_model(dropout=0.5)

# Classification loss
loss_fn = nn.CrossEntropyLoss()

print("Model created on:", device)
print("Output classes:", len(class_names))
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

In [ ]:
# Small initial search: 3 architectures × 2 learning rates
# plus an optimizer comparison on the original CNN.

experiments = [
    {
        "name": "original_adamw_lr0001",
        "architecture": "original_cnn",
        "optimizer": "AdamW",
        "lr": 1e-4,
    },
    {
        "name": "original_adamw_lr0003",
        "architecture": "original_cnn",
        "optimizer": "AdamW",
        "lr": 3e-4,
    },
    {
        "name": "original_sgd_lr001",
        "architecture": "original_cnn",
        "optimizer": "SGD",
        "lr": 1e-2,
    },
    {
        "name": "original_sgd_lr0001",
        "architecture": "original_cnn",
        "optimizer": "SGD",
        "lr": 1e-3,
    },
    {
        "name": "small_adamw_lr0001",
        "architecture": "small_cnn",
        "optimizer": "AdamW",
        "lr": 1e-4,
    },
    {
        "name": "small_adamw_lr0003",
        "architecture": "small_cnn",
        "optimizer": "AdamW",
        "lr": 3e-4,
    },
    {
        "name": "resnet18_adamw_lr0001",
        "architecture": "resnet18",
        "optimizer": "AdamW",
        "lr": 1e-4,
    },
    {
        "name": "resnet18_adamw_lr0003",
        "architecture": "resnet18",
        "optimizer": "AdamW",
        "lr": 3e-4,
    },
]

# Shared settings
BATCH_SIZE = 32
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4

# Scheduler settings: reduce LR when validation loss stalls
LR_FACTOR = 0.5
LR_PATIENCE = 1

print("Experiment configurations prepared:", len(experiments))
print("No training has started.")

for config in experiments:
    print(
        f"{config['architecture']:12s} | "
        f"{config['optimizer']:5s} | "
        f"learning rate = {config['lr']}"
    )

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# Pretrained weights and their matching image preprocessing
RESNET_WEIGHTS = ResNet18_Weights.DEFAULT
resnet_tf = RESNET_WEIGHTS.transforms()


# Architecture 2: smaller CNN, trained from scratch
def build_small_cnn(dropout=0.5):
    return nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),

        nn.AdaptiveAvgPool2d((4, 4)),
        nn.Flatten(),

        nn.Linear(64 * 4 * 4, 128),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(128, len(class_names))
    ).to(device)


# Architecture 3: frozen pretrained ResNet18 + new classifier
class FrozenResNet18(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()

        self.backbone = resnet18(
            weights=RESNET_WEIGHTS if pretrained else None
        )

        feature_count = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        # Freeze all pretrained parameters
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

        self.classifier = nn.Linear(feature_count, num_classes)
        self.backbone.eval()

    def train(self, mode=True):
        super().train(mode)

        # Keep BatchNorm statistics in the frozen backbone unchanged
        self.backbone.eval()
        return self

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)

        return self.classifier(features)


# Choose a model using the architecture name in our experiments
def create_model(architecture, pretrained=True):
    if architecture == "original_cnn":
        return build_model(dropout=DROPOUT)

    if architecture == "small_cnn":
        return build_small_cnn(dropout=DROPOUT)

    if architecture == "resnet18":
        return FrozenResNet18(
            num_classes=len(class_names),
            pretrained=pretrained
        ).to(device)

    raise ValueError(f"Unknown architecture: {architecture}")


# Check that all three models produce four class scores
for architecture in ["original_cnn", "small_cnn", "resnet18"]:
    print(f"\nChecking {architecture}...", flush=True)

    torch.manual_seed(SEED)
    check_model = create_model(architecture)
    check_model.eval()

    image_size = 224 if architecture == "resnet18" else 128
    dummy_image = torch.zeros(
        1, 3, image_size, image_size, device=device
    )

    with torch.no_grad():
        output = check_model(dummy_image)

    trainable = sum(
        p.numel() for p in check_model.parameters()
        if p.requires_grad
    )

    print("Output shape:", tuple(output.shape))
    print("Trainable parameters:", f"{trainable:,}")

    assert output.shape == (1, len(class_names))
    del check_model

print("\nAll three architectures passed. No training started.")

In [ ]:
from torch import optim

# 1. Give each architecture the correct preprocessing.
# All experiments still use the SAME image split.
def make_loaders(architecture):
    transform = resnet_tf if architecture == "resnet18" else tf

    dataset = datasets.ImageFolder(
        "data/Training",
        transform=transform
    )

    # Ensure our existing split indices still refer to the same files
    assert dataset.samples == full_train_ds.samples

    training_subset = Subset(dataset, train_indices)
    validation_subset = Subset(dataset, val_indices)

    # Reset image-shuffling seed for each experiment
    generator = torch.Generator().manual_seed(SEED)

    training_loader = DataLoader(
        training_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=generator
    )

    validation_loader = DataLoader(
        validation_subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    return training_loader, validation_loader


# 2. Create the optimizer specified by an experiment
def make_optimizer(model, config):
    trainable_parameters = [
        p for p in model.parameters() if p.requires_grad
    ]

    if config["optimizer"] == "AdamW":
        return optim.AdamW(
            trainable_parameters,
            lr=config["lr"],
            weight_decay=WEIGHT_DECAY
        )

    if config["optimizer"] == "SGD":
        return optim.SGD(
            trainable_parameters,
            lr=config["lr"],
            momentum=0.9,
            weight_decay=WEIGHT_DECAY
        )

    raise ValueError("Unknown optimizer: " + config["optimizer"])


# 3. Reduce LR when validation loss stops improving
def make_scheduler(optimizer):
    return optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        threshold=1e-4,
        min_lr=1e-6
    )


# 4. Check preprocessing using one validation batch per architecture
for architecture in ["original_cnn", "small_cnn", "resnet18"]:
    _, validation_loader = make_loaders(architecture)
    images, labels = next(iter(validation_loader))

    print(
        f"{architecture}: "
        f"image batch = {tuple(images.shape)}"
    )

print("\nData loaders, optimizer functions, and scheduler are ready.")
print("No training started.")

In [ ]:
import time
import os
from pathlib import Path

CHECKPOINT_DIR = Path("tuning_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)


# Run one complete training OR validation epoch
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    correct = 0
    total_images = 0
    start_time = time.perf_counter()

    for batch_number, (images, labels) in enumerate(loader, start=1):
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad(set_to_none=True)

        # Calculate gradients only during training
        with torch.set_grad_enabled(training):
            predictions = model(images)
            loss = loss_fn(predictions, labels)

            if not torch.isfinite(loss):
                raise RuntimeError("Loss became NaN or infinite.")

            if training:
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        correct += (
            predictions.argmax(dim=1) == labels
        ).sum().item()
        total_images += batch_size

        if batch_number == 1 or batch_number % 20 == 0:
            stage = "Training" if training else "Validation"
            print(
                f"  {stage}: batch {batch_number}/{len(loader)}",
                flush=True
            )

    return {
        "loss": total_loss / total_images,
        "accuracy": 100.0 * correct / total_images,
        "seconds": time.perf_counter() - start_time
    }


# Save enough information to resume after a completed epoch
def save_checkpoint(
    path, model, optimizer, scheduler,
    completed_epochs, config, history,
    best_val_loss, training_loader
):
    checkpoint = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),

        # Number of FULLY completed epochs, rather than epoch index
        "completed_epochs": completed_epochs,

        "config": dict(config),
        "history": list(history),
        "best_val_loss": best_val_loss,
        "class_names": list(class_names),

        "train_indices": list(train_indices),
        "val_indices": list(val_indices),
        "training_samples": list(full_train_ds.samples),

        "settings": {
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "dropout": DROPOUT,
            "weight_decay": WEIGHT_DECAY,
            "lr_factor": LR_FACTOR,
            "lr_patience": LR_PATIENCE
        },

        # Preserve random states for resuming training
        "torch_rng_state": torch.get_rng_state(),
        "python_rng_state": random.getstate(),
        "loader_rng_state": training_loader.generator.get_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available() else None
        )
    }

    # Finish writing a temporary file before replacing the checkpoint
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary_path)
    os.replace(temporary_path, path)


print("Training and validation functions ready.")
print("Checkpoint-saving function ready.")
print("Checkpoint folder:", CHECKPOINT_DIR.resolve())
print("No training started yet.")

In [ ]:
# First experiment: original CNN + AdamW + LR 0.0001
config = dict(experiments[0])

run_dir = CHECKPOINT_DIR / config["name"]
run_dir.mkdir(parents=True, exist_ok=True)

latest_path = run_dir / "latest.pth"
best_path = run_dir / "best.pth"

# Prevent accidentally overwriting an existing experiment
if latest_path.exists() or best_path.exists():
    raise RuntimeError(
        "A checkpoint already exists for this experiment. "
        "Use resume code instead of restarting it."
    )

# Fresh, repeatable initialization
random.seed(SEED)
torch.manual_seed(SEED)

model = create_model(config["architecture"])
optimizer = make_optimizer(model, config)
scheduler = make_scheduler(optimizer)

training_loader, validation_loader = make_loaders(
    config["architecture"]
)

history = []
best_val_loss = float("inf")

print("Experiment:", config["name"])
print("Device:", device)
print("Starting epoch 1...", flush=True)

epoch_start = time.perf_counter()
lr_used = optimizer.param_groups[0]["lr"]

# 1. Learn from training images
train_metrics = run_epoch(
    model, training_loader, optimizer
)

# 2. Evaluate on validation images
val_metrics = run_epoch(
    model, validation_loader
)

# 3. Update the learning-rate scheduler
scheduler.step(val_metrics["loss"])

# 4. Record results
completed_epochs = 1
best_val_loss = val_metrics["loss"]

history.append({
    "epoch": completed_epochs,
    "train_loss": train_metrics["loss"],
    "train_accuracy": train_metrics["accuracy"],
    "val_loss": val_metrics["loss"],
    "val_accuracy": val_metrics["accuracy"],
    "lr_used": lr_used,
    "next_lr": optimizer.param_groups[0]["lr"],
    "seconds": time.perf_counter() - epoch_start
})

# 5. First epoch is both the latest and best so far
for path in [latest_path, best_path]:
    save_checkpoint(
        path=path,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        completed_epochs=completed_epochs,
        config=config,
        history=history,
        best_val_loss=best_val_loss,
        training_loader=training_loader
    )

# 6. Verify that the latest checkpoint can be read
saved = torch.load(
    latest_path,
    map_location="cpu",
    weights_only=True
)
assert saved["completed_epochs"] == 1
del saved

print("\nEpoch 1 completed!")
print(f"Training loss: {train_metrics['loss']:.4f}")
print(f"Training accuracy: {train_metrics['accuracy']:.2f}%")
print(f"Validation loss: {val_metrics['loss']:.4f}")
print(f"Validation accuracy: {val_metrics['accuracy']:.2f}%")
print(f"Learning rate used: {lr_used}")
print(f"Time including saving: {(time.perf_counter() - epoch_start)/60:.2f} minutes")
print("\nBest checkpoint:", best_path)
print("Latest checkpoint:", latest_path)
print("Checkpoint reload check passed.")

In [ ]:
def train_experiment(config, total_epochs=5):
    config = dict(config)

    run_dir = CHECKPOINT_DIR / config["name"]
    run_dir.mkdir(parents=True, exist_ok=True)

    latest_path = run_dir / "latest.pth"
    best_path = run_dir / "best.pth"

    settings = {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "dropout": DROPOUT,
        "weight_decay": WEIGHT_DECAY,
        "lr_factor": LR_FACTOR,
        "lr_patience": LR_PATIENCE
    }

    random.seed(SEED)
    torch.manual_seed(SEED)

    training_loader, validation_loader = make_loaders(
        config["architecture"]
    )

    # When resuming, pretrained weights come from the checkpoint
    model = create_model(
        config["architecture"],
        pretrained=not latest_path.exists()
    )
    optimizer = make_optimizer(model, config)
    scheduler = make_scheduler(optimizer)

    completed_epochs = 0
    best_val_loss = float("inf")
    history = []

    if latest_path.exists():
        checkpoint = torch.load(
            latest_path,
            map_location="cpu",
            weights_only=True
        )

        # Prevent mixing different settings or data splits
        assert checkpoint["config"] == config, "Config changed."
        assert checkpoint["settings"] == settings, "Settings changed."
        assert checkpoint["class_names"] == list(class_names)
        assert checkpoint["train_indices"] == list(train_indices)
        assert checkpoint["val_indices"] == list(val_indices)
        assert checkpoint["training_samples"] == list(
            full_train_ds.samples
        ), "Dataset file list changed."

        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])

        completed_epochs = checkpoint["completed_epochs"]
        best_val_loss = checkpoint["best_val_loss"]
        history = checkpoint["history"]

        # Restore randomness AFTER constructing the model
        torch.set_rng_state(checkpoint["torch_rng_state"])
        random.setstate(checkpoint["python_rng_state"])
        training_loader.generator.set_state(
            checkpoint["loader_rng_state"]
        )

        if (
            torch.cuda.is_available()
            and checkpoint["cuda_rng_state"] is not None
        ):
            torch.cuda.set_rng_state_all(
                checkpoint["cuda_rng_state"]
            )

        del checkpoint

        print(
            f"Loaded checkpoint: {completed_epochs} epochs completed.",
            flush=True
        )

    elif best_path.exists():
        raise RuntimeError(
            "Found best.pth without latest.pth. "
            "Check the files before starting again."
        )

    if completed_epochs >= total_epochs:
        print(f"Already completed {total_epochs} or more epochs.")
        return history

    for epoch_number in range(completed_epochs + 1, total_epochs + 1):
        print(
            f"\n{config['name']} | Epoch {epoch_number}/{total_epochs}",
            flush=True
        )

        epoch_start = time.perf_counter()
        lr_used = optimizer.param_groups[0]["lr"]

        train_metrics = run_epoch(
            model, training_loader, optimizer
        )
        val_metrics = run_epoch(
            model, validation_loader
        )

        scheduler.step(val_metrics["loss"])
        next_lr = optimizer.param_groups[0]["lr"]

        improved = val_metrics["loss"] < best_val_loss
        if improved:
            best_val_loss = val_metrics["loss"]

        history.append({
            "epoch": epoch_number,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "lr_used": lr_used,
            "next_lr": next_lr,
            "seconds": time.perf_counter() - epoch_start
        })

        # Save the best model only when validation loss improves
        paths_to_save = [best_path, latest_path] if improved else [
            latest_path
        ]

        for path in paths_to_save:
            save_checkpoint(
                path=path,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                completed_epochs=epoch_number,
                config=config,
                history=history,
                best_val_loss=best_val_loss,
                training_loader=training_loader
            )

        print(
            f"Train accuracy: {train_metrics['accuracy']:.2f}% | "
            f"Validation accuracy: {val_metrics['accuracy']:.2f}%",
            flush=True
        )
        print(
            f"Validation loss: {val_metrics['loss']:.4f} | "
            f"LR used: {lr_used:g} | Next LR: {next_lr:g}",
            flush=True
        )
        print(
            "Latest checkpoint saved."
            + (" Best checkpoint updated." if improved else ""),
            flush=True
        )

    return history


print("Training function ready. Supports automatic resume.")

In [ ]:
# Continue experiment 1 up to FIVE total epochs
baseline_history = train_experiment(
    experiments[0],
    total_epochs=5
)

In [ ]:
# Run every configuration to 5 total epochs.
# The completed first experiment will be skipped automatically.

all_histories = {}

for number, config in enumerate(experiments, start=1):
    print(
        f"\n===== EXPERIMENT {number}/{len(experiments)}: "
        f"{config['name']} =====",
        flush=True
    )

    all_histories[config["name"]] = train_experiment(
        config,
        total_epochs=5
    )

print("\nALL EIGHT EXPERIMENTS COMPLETED.")
print("Checkpoints saved. Ready for comparison and evaluation.")

In [ ]:
all_histories = {}

for number, config in enumerate(experiments, start=1):
    print(
        f"\n===== EXPERIMENT {number}/{len(experiments)}: "
        f"{config['name']} =====",
        flush=True
    )
    all_histories[config["name"]] = train_experiment(
        config, total_epochs=5
    )

print("\nALL EIGHT EXPERIMENTS COMPLETED.")

In [ ]:
from IPython.display import display, HTML
from html import escape

# The checkpoint results were already loaded before the error.
comparison_rows = sorted(
    rows, key=lambda row: row["Validation loss"]
)

columns = list(comparison_rows[0])

table = "<table border='1' cellpadding='6'><tr>"
table += "".join(f"<th>{escape(c)}</th>" for c in columns)
table += "</tr>"

for row in comparison_rows:
    table += "<tr>"
    for column in columns:
        value = row[column]
        text = f"{value:.4f}" if isinstance(value, float) else str(value)
        table += f"<td>{escape(text)}</td>"
    table += "</tr>"

table += "</table>"
display(HTML(table))

best_name = comparison_rows[0]["Experiment"]
best_config = next(
    config for config in experiments
    if config["name"] == best_name
)

print("\nSelected experiment:", best_name)

In [ ]:
import hashlib
from PIL import Image
from collections import defaultdict
from itertools import combinations

split_samples = {
    "Training": [full_train_ds.samples[i] for i in train_indices],
    "Validation": [full_train_ds.samples[i] for i in val_indices],
    "Testing": test_ds.samples,
}

image_hashes = {}

for split_name, samples in split_samples.items():
    hashes = defaultdict(list)

    for path, label in samples:
        with Image.open(path) as image:
            image = image.convert("RGB")
            digest = hashlib.sha256(
                str(image.size).encode() + image.tobytes()
            ).hexdigest()

        hashes[digest].append((path, label))

    image_hashes[split_name] = hashes
    print(f"{split_name}: checked {len(samples)} images.")

duplicate_overlaps = {}

for first, second in combinations(split_samples, 2):
    shared = set(image_hashes[first]) & set(image_hashes[second])
    duplicate_overlaps[(first, second)] = shared

    print(
        f"\n{first} / {second}: "
        f"{len(shared)} shared exact-image groups"
    )

    for digest in sorted(shared)[:3]:
        print("  ", image_hashes[first][digest][0][0])
        print("  ", image_hashes[second][digest][0][0])

print("\nDuplicate check complete.")

In [ ]:
from torch.utils.data import Subset
from collections import Counter, defaultdict

# Check whether identical images have conflicting class labels.
labels_by_hash = defaultdict(set)

for hashes in image_hashes.values():
    for digest, records in hashes.items():
        labels_by_hash[digest].update(label for _, label in records)

conflicts = [
    digest for digest, labels in labels_by_hash.items()
    if len(labels) > 1
]

if conflicts:
    raise ValueError(
        f"{len(conflicts)} identical-image groups have conflicting labels. "
        "Send me this output before continuing."
    )

train_hashes = set(image_hashes["Training"])
original_val_hashes = set(image_hashes["Validation"])

# Keep one copy of each validation image not seen during training.
clean_val_paths = {
    records[0][0]
    for digest, records in image_hashes["Validation"].items()
    if digest not in train_hashes
}

# Exclude anything present in either original development split.
clean_test_paths = {
    records[0][0]
    for digest, records in image_hashes["Testing"].items()
    if digest not in (train_hashes | original_val_hashes)
}

clean_val_indices = [
    i for i in val_indices
    if full_train_ds.samples[i][0] in clean_val_paths
]

clean_test_indices = [
    i for i, (path, _) in enumerate(test_ds.samples)
    if path in clean_test_paths
]

clean_val_ds = Subset(full_train_ds, clean_val_indices)
clean_test_ds = Subset(test_ds, clean_test_indices)

for name, dataset in [
    ("Filtered validation", clean_val_ds),
    ("Filtered testing", clean_test_ds),
]:
    counts = Counter(
        dataset.dataset.samples[i][1] for i in dataset.indices
    )
    print(f"\n{name}: {len(dataset)} images")
    for class_id, class_name in enumerate(class_names):
        print(f"  {class_name}: {counts[class_id]}")
    assert all(counts[i] > 0 for i in range(len(class_names)))

print("\nEvaluation subsets ready. No files deleted.")

In [ ]:
import torch
from pathlib import Path
from torchvision import datasets
from torch.utils.data import DataLoader, Subset

filtered_results = []

for number, config in enumerate(experiments, start=1):
    print(f"\nEvaluating {number}/8: {config['name']}", flush=True)

    # Use the preprocessing required by each architecture.
    transform = resnet_tf if config["architecture"] == "resnet18" else tf
    dataset = datasets.ImageFolder("data/Training", transform=transform)
    assert dataset.samples == full_train_ds.samples

    loader = DataLoader(
        Subset(dataset, clean_val_indices),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    path = Path("tuning_checkpoints") / config["name"] / "best.pth"
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    assert checkpoint["class_names"] == list(class_names)

    candidate = create_model(config["architecture"], pretrained=False)
    candidate.load_state_dict(checkpoint["model_state"])

    metrics = run_epoch(candidate, loader)

    filtered_results.append({
        "name": config["name"],
        "val_loss": metrics["loss"],
        "val_accuracy": metrics["accuracy"],
        "checkpoint_epoch": checkpoint["completed_epochs"]
    })

    del candidate, checkpoint

filtered_results.sort(key=lambda result: result["val_loss"])

print("\nFILTERED VALIDATION RESULTS")
print(f"{'Experiment':<30} {'Loss':>10} {'Accuracy':>12}")

for result in filtered_results:
    print(
        f"{result['name']:<30} "
        f"{result['val_loss']:>10.4f} "
        f"{result['val_accuracy']:>11.2f}%"
    )

best_name = filtered_results[0]["name"]
best_config = next(
    config for config in experiments if config["name"] == best_name
)

print("\nSelected experiment:", best_name)

In [ ]:
needed = [
    "experiments", "create_model", "run_epoch",
    "clean_val_indices", "full_train_ds",
    "tf", "resnet_tf", "BATCH_SIZE", "loss_fn", "device"
]

missing = [name for name in needed if name not in globals()]

print("Missing setup variables:", missing)
print("Available evaluation results:",
      len(globals().get("filtered_results", [])))

In [ ]:
import json
from pathlib import Path

notebook_path = Path("brain_tumor_tuning.ipynb")
notebook = json.loads(notebook_path.read_text(encoding="utf-8"))

# Restore only the inspected setup and duplicate-filtering cells.
restore_cells = [
    (1, "data_dir ="),
    (2, "SEED = 42"),
    (3, "def build_model"),
    (4, "experiments ="),
    (5, "class FrozenResNet18"),
    (6, "def make_loaders"),
    (7, "def run_epoch"),
    (14, "image_hashes ="),
    (15, "clean_val_indices ="),
]

# Verify cell positions before running anything.
for number, expected_text in restore_cells:
    source = "".join(notebook["cells"][number - 1]["source"])
    assert expected_text in source, (
        f"Cell {number} has moved. Stop and send me this message."
    )

for number, _ in restore_cells:
    print(f"\nRestoring setup cell {number}...", flush=True)
    source = "".join(notebook["cells"][number - 1]["source"])
    exec(compile(source, f"setup_cell_{number}", "exec"), globals())

print("\nSETUP RESTORED. No training was run.")

In [ ]:
import json
import csv
import torch
from pathlib import Path
from torchvision import datasets
from torch.utils.data import DataLoader, Subset

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Save the completed validation comparison.
(results_dir / "filtered_validation_results.json").write_text(
    json.dumps(filtered_results, indent=2), encoding="utf-8"
)

# Load the selected model.
best_path = Path("tuning_checkpoints") / best_name / "best.pth"
best_checkpoint = torch.load(
    best_path, map_location="cpu", weights_only=True
)
assert best_checkpoint["class_names"] == list(class_names)

best_model = create_model(best_config["architecture"], pretrained=False)
best_model.load_state_dict(best_checkpoint["model_state"])
best_model.eval()

# Match preprocessing to the selected architecture.
test_transform = (
    resnet_tf if best_config["architecture"] == "resnet18" else tf
)
evaluation_dataset = datasets.ImageFolder(
    "data/Testing", transform=test_transform
)
assert evaluation_dataset.samples == test_ds.samples

evaluation_loader = DataLoader(
    Subset(evaluation_dataset, clean_test_indices),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

y_true, y_pred = [], []
test_loss_sum = 0.0

with torch.inference_mode():
    for batch_number, (images, labels) in enumerate(
        evaluation_loader, start=1
    ):
        outputs = best_model(images.to(device))
        test_loss_sum += (
            loss_fn(outputs, labels.to(device)).item() * labels.size(0)
        )
        y_true.extend(labels.tolist())
        y_pred.extend(outputs.argmax(dim=1).cpu().tolist())

        if batch_number == 1 or batch_number % 10 == 0:
            print(
                f"Testing batch {batch_number}/{len(evaluation_loader)}",
                flush=True
            )

# Rows = true classes; columns = predicted classes.
cm = torch.zeros(
    (len(class_names), len(class_names)), dtype=torch.int64
)
for actual, predicted in zip(y_true, y_pred):
    cm[actual, predicted] += 1

tp = cm.diag().double()
support = cm.sum(dim=1)
precision = tp / cm.sum(dim=0).clamp(min=1)
recall = tp / support.clamp(min=1)
f1 = 2 * precision * recall / (precision + recall).clamp(min=1e-12)
test_accuracy = 100 * tp.sum().item() / len(y_true)

print(f"\nFILTERED TEST ACCURACY: {test_accuracy:.2f}%")
print(f"{'Class':<15} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Images':>8}")

per_class = {}
for i, name in enumerate(class_names):
    per_class[name] = {
        "precision": precision[i].item(),
        "recall": recall[i].item(),
        "f1": f1[i].item(),
        "support": support[i].item()
    }
    print(
        f"{name:<15} {precision[i]:>10.3f} "
        f"{recall[i]:>10.3f} {f1[i]:>10.3f} {support[i]:>8}"
    )

print(f"\nMacro F1: {f1.mean().item():.3f}")

test_report = {
    "experiment": best_name,
    "test_images": len(y_true),
    "accuracy_percent": test_accuracy,
    "loss": test_loss_sum / len(y_true),
    "macro_f1": f1.mean().item(),
    "per_class": per_class,
    "class_names": list(class_names),
    "confusion_matrix": cm.tolist(),
    "limitation": (
        "Exact duplicates filtered after training. Original validation "
        "influenced training decisions. Patient overlap and near-duplicates "
        "have not been ruled out."
    )
}
(results_dir / "filtered_test_report.json").write_text(
    json.dumps(test_report, indent=2), encoding="utf-8"
)

with (results_dir / "test_predictions.csv").open(
    "w", newline="", encoding="utf-8"
) as file:
    writer = csv.writer(file)
    writer.writerow(["image", "true_class", "predicted_class"])
    for index, actual, predicted in zip(
        clean_test_indices, y_true, y_pred
    ):
        writer.writerow([
            evaluation_dataset.samples[index][0],
            class_names[actual],
            class_names[predicted]
        ])

print("\nResults saved in:", results_dir.resolve())

In [ ]:
import matplotlib.pyplot as plt
import torch

# 1. Confusion matrix
matrix = cm.cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
plot = ax.imshow(matrix, cmap="Blues")
fig.colorbar(plot, ax=ax)

ax.set(
    xticks=range(len(class_names)),
    yticks=range(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel="Predicted class",
    ylabel="Actual class",
    title="Confusion matrix — filtered test set"
)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(
            j, i, str(matrix[i, j]),
            ha="center", va="center",
            color="white" if matrix[i, j] > matrix.max() / 2 else "black"
        )

fig.tight_layout()
fig.savefig(results_dir / "confusion_matrix.png", dpi=200)
plt.show()

# 2. Full training history of the selected experiment
latest_checkpoint = torch.load(
    best_path.parent / "latest.pth",
    map_location="cpu",
    weights_only=True
)
history = latest_checkpoint["history"]
epochs = [row["epoch"] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(
    epochs, [row["train_loss"] for row in history],
    "o-", label="Training"
)
axes[0].plot(
    epochs, [row["val_loss"] for row in history],
    "o-", label="Original validation"
)
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss curves")

axes[1].plot(
    epochs, [row["train_accuracy"] for row in history],
    "o-", label="Training"
)
axes[1].plot(
    epochs, [row["val_accuracy"] for row in history],
    "o-", label="Original validation"
)
axes[1].set(
    xlabel="Epoch", ylabel="Accuracy (%)", title="Accuracy curves"
)

for ax in axes:
    ax.set_xticks(epochs)
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle(
    "Recorded training history — validation duplicates were filtered later",
    fontsize=11
)
fig.tight_layout()
fig.savefig(results_dir / "training_curves.png", dpi=200)
plt.show()

print("Both plots saved in:", results_dir.resolve())

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

# 1. Show four correct and four incorrect predictions.
sample_rng = random.Random(42)

correct_indices = [
    i for i, (actual, predicted) in enumerate(zip(y_true, y_pred))
    if actual == predicted
]
incorrect_indices = [
    i for i, (actual, predicted) in enumerate(zip(y_true, y_pred))
    if actual != predicted
]

fig, axes = plt.subplots(2, 4, figsize=(12, 7))

for row, candidates in enumerate([correct_indices, incorrect_indices]):
    selected = sample_rng.sample(candidates, min(4, len(candidates)))

    for ax in axes[row]:
        ax.axis("off")

    for ax, position in zip(axes[row], selected):
        dataset_index = clean_test_indices[position]
        path = evaluation_dataset.samples[dataset_index][0]

        with Image.open(path) as image:
            ax.imshow(image.convert("RGB"))

        ax.set_title(
            f"Actual: {class_names[y_true[position]]}\n"
            f"Predicted: {class_names[y_pred[position]]}",
            color="green" if row == 0 else "red",
            fontsize=10
        )

fig.suptitle(
    "Filtered test examples: correct (top), incorrect (bottom)"
)
fig.tight_layout()
fig.savefig(results_dir / "prediction_examples.png", dpi=200)
plt.show()

# 2. Compare all eight models using filtered validation loss.
ordered = sorted(filtered_results, key=lambda row: row["val_loss"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    [row["name"] for row in ordered],
    [row["val_loss"] for row in ordered],
    color=["seagreen"] + ["steelblue"] * (len(ordered) - 1)
)
ax.invert_yaxis()
ax.set(
    xlabel="Filtered validation loss — lower is better",
    title="Comparison of eight saved models"
)
fig.tight_layout()
fig.savefig(results_dir / "model_comparison.png", dpi=200)
plt.show()

print("Sample predictions and model comparison saved.")

In [ ]:
from pathlib import Path
import torch

export_dir = Path("exported_model")
export_dir.mkdir(exist_ok=True)
export_path = export_dir / "brain_tumor_model.pth"

assert best_config["architecture"] == "original_cnn"

# Save weights and the information needed for prediction.
export_data = {
    "model_state": {
        name: value.detach().cpu().clone()
        for name, value in best_model.state_dict().items()
    },
    "architecture": best_config["architecture"],
    "experiment": best_name,
    "class_names": list(class_names),
    "dropout": DROPOUT,
    "preprocessing": {
        "color_mode": "RGB",
        "image_size": [128, 128],
        "mean": [0.5, 0.5, 0.5],
        "std": [0.5, 0.5, 0.5]
    },
    "evaluation": test_report,
    "intended_use": "Educational research; not for clinical diagnosis."
}

torch.save(export_data, export_path)

# Reload and check that its outputs match the evaluated model.
saved = torch.load(
    export_path, map_location="cpu", weights_only=True
)
reloaded_model = create_model(
    saved["architecture"], pretrained=False
)
reloaded_model.load_state_dict(saved["model_state"])
reloaded_model.eval()
best_model.eval()

sample_images, _ = next(iter(evaluation_loader))
sample_images = sample_images[:2].to(device)

with torch.inference_mode():
    original_output = best_model(sample_images)
    reloaded_output = reloaded_model(sample_images)

torch.testing.assert_close(original_output, reloaded_output)
del reloaded_model

print("MODEL EXPORTED AND RELOAD CHECK PASSED")
print("Saved at:", export_path.resolve())
print(f"File size: {export_path.stat().st_size / (1024**2):.1f} MB")

In [ ]:
from pathlib import Path
import textwrap
import subprocess
import sys
import json

prediction_code = r'''
import argparse
import json
from pathlib import Path

import torch
from torch import nn
from torchvision import transforms
from PIL import Image


def predict(image_path, model_path):
    checkpoint = torch.load(
        model_path, map_location="cpu", weights_only=True
    )
    if checkpoint["architecture"] != "original_cnn":
        raise ValueError("This script expects the exported original CNN.")

    classes = checkpoint["class_names"]

    model = nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(128 * 16 * 16, 256),
        nn.ReLU(),
        nn.Dropout(checkpoint["dropout"]),
        nn.Linear(256, len(classes))
    )
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    settings = checkpoint["preprocessing"]
    transform = transforms.Compose([
        transforms.Resize(tuple(settings["image_size"])),
        transforms.ToTensor(),
        transforms.Normalize(settings["mean"], settings["std"])
    ])

    with Image.open(image_path) as image:
        inputs = transform(
            image.convert(settings["color_mode"])
        ).unsqueeze(0)

    with torch.inference_mode():
        scores = model(inputs).softmax(dim=1)[0]

    predicted_index = scores.argmax().item()
    return {
        "predicted_class": classes[predicted_index],
        "softmax_scores": {
            name: float(scores[i]) for i, name in enumerate(classes)
        },
        "note": (
            "Scores are not calibrated clinical probabilities. "
            "Educational use only; not for diagnosis."
        )
    }


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("image", type=Path)
    parser.add_argument(
        "--model",
        type=Path,
        default=Path(__file__).resolve().parent
        / "exported_model" / "brain_tumor_model.pth"
    )
    args = parser.parse_args()
    print(json.dumps(predict(args.image, args.model), indent=2))
'''

script_path = Path("predict_brain_tumor.py")
script_path.write_text(
    textwrap.dedent(prediction_code).lstrip(),
    encoding="utf-8"
)

# Run the script in a separate Python process.
example_path = evaluation_dataset.samples[clean_test_indices[0]][0]

completed = subprocess.run(
    [sys.executable, str(script_path), example_path],
    capture_output=True,
    text=True
)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("Standalone prediction failed.")

prediction = json.loads(completed.stdout)
assert prediction["predicted_class"] == class_names[y_pred[0]], (
    "Standalone prediction differs from the notebook prediction."
)

print(completed.stdout)
print("STANDALONE PREDICTION CHECK PASSED")
print("Script saved:", script_path.resolve())

In [ ]:
from pathlib import Path
from importlib.metadata import version
import shutil
import json
import subprocess
import platform

source = Path.cwd()
project = source / "brain-tumor-classifier"
project.mkdir(exist_ok=True)

# Copy the tested prediction script and exported model.
shutil.copy2(
    source / "predict_brain_tumor.py",
    project / "predict_brain_tumor.py"
)
(project / "exported_model").mkdir(exist_ok=True)
shutil.copy2(
    source / "exported_model" / "brain_tumor_model.pth",
    project / "exported_model" / "brain_tumor_model.pth"
)

# Keep the notebook as an experiment record.
(project / "notebooks").mkdir(exist_ok=True)
notebook = json.loads(
    (source / "brain_tumor_tuning.ipynb").read_text(encoding="utf-8")
)

# Remove screenshot-style outputs and local paths from the shared copy.
# The original notebook is unchanged.
for cell in notebook["cells"]:
    if cell["cell_type"] == "code":
        cell["outputs"] = []
        cell["execution_count"] = None

(project / "notebooks" / "experiment_log.ipynb").write_text(
    json.dumps(notebook, indent=1), encoding="utf-8"
)

# Copy metrics and plots that do not contain dataset images.
(project / "results").mkdir(exist_ok=True)
for filename in [
    "filtered_validation_results.json",
    "filtered_test_report.json",
    "confusion_matrix.png",
    "training_curves.png",
    "model_comparison.png",
]:
    shutil.copy2(source / "results" / filename, project / "results" / filename)

packages = ["torch", "torchvision", "Pillow", "matplotlib"]
requirements = "\n".join(
    f"{package}=={version(package)}" for package in packages
) + "\n"
(project / "requirements.txt").write_text(requirements, encoding="utf-8")

(project / ".gitignore").write_text(
    """__pycache__/
*.pyc
.ipynb_checkpoints/
.venv/
venv/
.env
data/
tuning_checkpoints/
*.tmp
""",
    encoding="utf-8"
)

report = json.loads(
    (project / "results" / "filtered_test_report.json").read_text(
        encoding="utf-8"
    )
)

readme = f"""# Brain Tumor Classifier

A PyTorch project comparing CNN architectures and transfer learning for
four-class brain MRI classification: glioma, meningioma, no tumour,
and pituitary tumour.

## Experiments

- Eight configurations, each trained for five epochs.
- Original CNN, smaller CNN, and pretrained ResNet18 with a frozen backbone.
- AdamW and SGD optimizer comparisons and different learning rates.
- Dropout, weight decay, and ReduceLROnPlateau learning-rate scheduling.
- Best-model and resumable training checkpoints.
- Model selection using filtered validation loss.

## Results

Selected experiment: `{report['experiment']}`

- Filtered validation accuracy: 88.63%.
- Filtered test images: {report['test_images']}.
- Filtered test accuracy: {report['accuracy_percent']:.2f}%.
- Macro F1: {report['macro_f1']:.3f}.

Per-class metrics are available in `results/filtered_test_report.json`.

![Confusion matrix](results/confusion_matrix.png)
![Model comparison](results/model_comparison.png)
![Training curves](results/training_curves.png)

## Evaluation limitations

Exact-image duplicates were discovered after training:
47 shared groups between training and validation, 85 between training
and testing, and 26 between validation and testing. These counts
are pairwise and should not be added as unique-image counts.

Validation images identical to training images were excluded.
Test images identical to either original development split were excluded.
Repeated images within the remaining evaluation subsets were reduced
to one copy.

Filtering was retrospective: the original validation set had already
influenced checkpoint selection and learning-rate scheduling.
The training curves show that original validation history.
Patient overlap and near-duplicates have not been ruled out.
These are exploratory results, not a fully leakage-free benchmark.

## Run a prediction

The exported model must be present at:
`exported_model/brain_tumor_model.pth`

Install dependencies:

```bash
python -m pip install -r requirements.txt